<a href="https://colab.research.google.com/github/Yiiize/MSSP6070/blob/main/Assignments/Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import userdata
import os

github_token = userdata.get('Git_Key')
owner = 'Yiiize' # Replace with the GitHub repository owner
repository = 'MSSP6070' # Replace with the GitHub repository name

clone_url = f'https://{github_token}@github.com/{owner}/{repository}.git'

# Clone the repository
import subprocess
subprocess.run(['git', 'clone', clone_url])

# Navigate into the cloned repository directory (optional, but often useful)
os.chdir(repository)
print(f"Changed directory to: {os.getcwd()}")


Changed directory to: /content/MSSP6070


In [ ]:
# =========================================
# Academic Performance Analysis (Colab-ready)
# =========================================
# If needed in Colab, uncomment:
# !pip install pandas numpy scipy statsmodels openpyxl matplotlib

import os, re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# -------- Options --------
USE_MENDELEY_CSV = False  # Set True if using the CSV instead of Excel
DATA_EXCEL_PATH = "/content/MSSP6070/data/data_academic_performance.xlsx"  # <-- change if needed
DATA_EXCEL_SHEET = "SABER11_SABERPRO"                        # <-- sheet name in your file
MENDELEY_CSV_PATH = "/content/SABER11_SABERPRO.csv"          # <-- set if using CSV

# Where to save outputs
OUT_DIR = Path("/content/edu_analysis_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================
# 1) Load data
# =========================================
if USE_MENDELEY_CSV:
    df = pd.read_csv(MENDELEY_CSV_PATH)
else:
    df = pd.read_excel(DATA_EXCEL_PATH, sheet_name=DATA_EXCEL_SHEET)

# Normalize column names to UPPER
df.columns = [c.strip().upper() for c in df.columns]

print("Rows:", df.shape[0], "Cols:", df.shape[1])
print("First 20 columns:", df.columns[:20].tolist())

# =========================================
# 2) Identify columns & construct metrics
# =========================================
PRO_COLS_CAND = ["CR_PRO","QR_PRO","CC_PRO","WC_PRO","ENG_PRO"]
S11_COLS_CAND = ["MAT_S11","CR_S11","CC_S11","BIO_S11","ENG_S11"]

pro_cols = [c for c in PRO_COLS_CAND if c in df.columns]
s11_cols = [c for c in S11_COLS_CAND if c in df.columns]

# Global professional score (prefer provided G_SC)
if "G_SC" in df.columns:
    df["_GLOBAL_PRO"] = pd.to_numeric(df["G_SC"], errors="coerce")
else:
    if len(pro_cols) == 0:
        raise ValueError("No professional-stage score columns (CR/QR/CC/WC/ENG_PRO) or G_SC found.")
    df["_GLOBAL_PRO"] = df[pro_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# Global S11 composite if available
df["_GLOBAL_S11"] = np.nan
if len(s11_cols) > 0:
    df["_GLOBAL_S11"] = df[s11_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

# Normalize gender to {M, F} as possible
if "GENDER" in df.columns:
    df["GENDER"] = (
        df["GENDER"].astype(str).str.strip().str.upper()
          .replace({"FEMALE":"F","FEMENINO":"F","MUJER":"F","WOMAN":"F",
                    "MALE":"M","MASCULINO":"M","HOMBRE":"M","MAN":"M"})
    )

# Parent education mapping → ordinal
def map_parent_edu(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().upper()
    mapping = {
        'NINGUNO':0,'NONE':0,'NO FORMAL':0,
        'PRIMARIA':1,'PRIMARY':1,
        'SECUNDARIA':2,'SECONDARY':2,'BACHILLERATO':2,'HIGH SCHOOL':2,
        'TECNICO':3,'TECHNICAL':3,'TECNOLÓGICO':3,'TECHNOLOGICAL':3,
        'UNIVERSIDAD':4,'COLLEGE':4,'UNIVERSITARIO':4,'BACHELOR':4,
        'POSGRADO':5,'POSTGRADUATE':5,'MAESTRIA':5,'MASTERS':5,'ESPECIALIZACION':5,
        'DOCTORADO':6,'PHD':6
    }
    if s in mapping:
        return mapping[s]
    for k,v in mapping.items():
        if k in s:
            return v
    try:
        return float(s)
    except:
        return np.nan

for c in ["EDU_FATHER","EDU_MOTHER"]:
    if c in df.columns:
        df[c+"_ORD"] = df[c].map(map_parent_edu)

parent_ord_cols = [c for c in ["EDU_FATHER_ORD","EDU_MOTHER_ORD"] if c in df.columns]
df["_PARENT_EDU_MAX"] = df[parent_ord_cols].max(axis=1) if parent_ord_cols else np.nan

# =========================================
# 3) Summaries by group (Pandas 2.x safe)
# =========================================
def group_summary(data, group_col, target_cols):
    present = [c for c in target_cols if c in data.columns]
    if len(present) == 0:
        return None
    return data.groupby(group_col)[present].agg(["count","mean","std"])

# By Gender
summ_gender = None
if "GENDER" in df.columns:
    summ_gender = group_summary(df, "GENDER", ["_GLOBAL_PRO","_GLOBAL_S11"])
    if summ_gender is not None:
        summ_gender.to_csv(OUT_DIR/"summary_by_gender.csv")

# By Stratum (cast to string for grouping/reporting; keep numeric for tests)
summ_stratum = None
if "STRATUM" in df.columns:
    df["STRATUM_LABEL"] = df["STRATUM"].astype(str)
    summ_stratum = group_summary(df, "STRATUM_LABEL", ["_GLOBAL_PRO","_GLOBAL_S11"])
    if summ_stratum is not None:
        summ_stratum.to_csv(OUT_DIR/"summary_by_strata.csv")

# By Parental Education bands
bins = [-0.1,0.5,1.5,2.5,3.5,4.5,5.5,10]
labels = ['None','Primary','Secondary/HS','Technical','College/BA','Postgrad','Higher']
df["_PARENT_BAND"] = pd.cut(df["_PARENT_EDU_MAX"], bins=bins, labels=labels, include_lowest=True)
summ_parent = df.groupby("_PARENT_BAND")[["_GLOBAL_PRO","_GLOBAL_S11"]].agg(["count","mean","std"])
summ_parent.to_csv(OUT_DIR/"summary_by_parent_education.csv")

print("Saved summaries:",
      [p.name for p in OUT_DIR.glob("summary_*.csv")])

# =========================================
# 4) Statistical tests
# =========================================
results = {}

# Welch t-test (gender)
if "GENDER" in df.columns:
    g = df[["GENDER","_GLOBAL_PRO"]].dropna()
    if g["GENDER"].nunique() == 2:
        a = g.loc[g["GENDER"] == "M", "_GLOBAL_PRO"]
        b = g.loc[g["GENDER"] == "F", "_GLOBAL_PRO"]
        tstat, pval = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
        results["gender_ttest"] = {
            "n_M": int(a.shape[0]),
            "n_F": int(b.shape[0]),
            "mean_M": float(np.nanmean(a)),
            "mean_F": float(np.nanmean(b)),
            "diff_M_minus_F": float(np.nanmean(a) - np.nanmean(b)),
            "t": float(tstat), "p": float(pval)
        }

# One-way ANOVA (strata)
if "STRATUM" in df.columns:
    s = df[["STRATUM","_GLOBAL_PRO"]].dropna()
    if s["STRATUM"].nunique() >= 2:
        groups = [vals["_GLOBAL_PRO"].values for _, vals in s.groupby("STRATUM")]
        F, p = stats.f_oneway(*groups)
        results["strata_anova"] = {
            "k_groups": int(len(groups)),
            "n_total": int(s.shape[0]),
            "F": float(F), "p": float(p)
        }

pd.Series(results).to_json(OUT_DIR/"hypothesis_tests.json")
print("Saved:", OUT_DIR/"hypothesis_tests.json")

# =========================================
# 5) Regression (HC3): G_SC ~ parent_edu + S11 + gender + stratum
# =========================================
reg_cols = ["_GLOBAL_PRO","_PARENT_EDU_MAX","_GLOBAL_S11"]
if "GENDER" in df.columns: reg_cols.append("GENDER")
if "STRATUM" in df.columns: reg_cols.append("STRATUM")

reg_df = df[reg_cols].dropna()
print("Regression N:", reg_df.shape[0])

if reg_df.shape[0] > 50:
    formula = "_GLOBAL_PRO ~ _PARENT_EDU_MAX"
    if "_GLOBAL_S11" in reg_df.columns: formula += " + _GLOBAL_S11"
    if "GENDER" in reg_df.columns:       formula += " + C(GENDER)"
    if "STRATUM" in reg_df.columns:      formula += " + C(STRATUM)"

    model = smf.ols(formula, data=reg_df).fit(cov_type="HC3")
    coef_tbl = model.summary2().tables[1]
    coef_tbl.to_csv(OUT_DIR/"regression_coefficients.csv")

    with open(OUT_DIR/"regression_summary.txt","w") as f:
        f.write(model.summary().as_text())

    print("Saved regression tables.")
else:
    print("Not enough rows for regression with the requested controls.")

# =========================================
# 6) Simple plots (no style/colors set)
# =========================================
# Mean by gender
if summ_gender is not None:
    try:
        g_means = (summ_gender["_GLOBAL_PRO"]["mean"]
                   .reset_index()
                   .rename(columns={"mean":"GLOBAL_PRO_MEAN"}))
        plt.figure()
        plt.bar(g_means["GENDER"].astype(str), g_means["GLOBAL_PRO_MEAN"])
        plt.title("Global Professional Score (G_SC) by Gender")
        plt.xlabel("Gender"); plt.ylabel("Mean G_SC")
        plt.tight_layout()
        plt.savefig(OUT_DIR/"plot_mean_by_gender.png", dpi=150)
        plt.close()
    except Exception as e:
        print("Plot gender means warning:", e)

# Mean by stratum (order numerically if we can parse trailing number)
if summ_stratum is not None:
    try:
        s_means = (summ_stratum["_GLOBAL_PRO"]["mean"]
                   .reset_index()
                   .rename(columns={"STRATUM_LABEL":"STRATUM","mean":"GLOBAL_PRO_MEAN"}))
        def to_num_like(x):
            # try to pull a number from end (e.g., "Stratum 4" -> 4)
            parts = str(x).split()
            try:
                return float(parts[-1])
            except:
                return np.nan
        s_means["_ORD"] = s_means["STRATUM"].apply(to_num_like)
        s_means = s_means.sort_values(["_ORD","STRATUM"])
        plt.figure()
        plt.bar(s_means["STRATUM"].astype(str), s_means["GLOBAL_PRO_MEAN"])
        plt.xticks(rotation=45, ha="right")
        plt.title("Global Professional Score (G_SC) by Socioeconomic Stratum")
        plt.xlabel("Stratum"); plt.ylabel("Mean G_SC")
        plt.tight_layout()
        plt.savefig(OUT_DIR/"plot_mean_by_stratum.png", dpi=150)
        plt.close()
    except Exception as e:
        print("Plot stratum means warning:", e)

print("All done! Outputs saved in:", OUT_DIR.resolve())


Rows: 12411 Cols: 45
First 20 columns: ['COD_S11', 'GENDER', 'EDU_FATHER', 'EDU_MOTHER', 'OCC_FATHER', 'OCC_MOTHER', 'STRATUM', 'SISBEN', 'PEOPLE_HOUSE', 'UNNAMED: 9', 'INTERNET', 'TV', 'COMPUTER', 'WASHING_MCH', 'MIC_OVEN', 'CAR', 'DVD', 'FRESH', 'PHONE', 'MOBILE']
Saved summaries: ['summary_by_parent_education.csv', 'summary_by_gender.csv', 'summary_by_strata.csv']
Saved: /content/edu_analysis_outputs/hypothesis_tests.json
Regression N: 4561


/tmp/ipython-input-1247504519.py:127: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summ_parent = df.groupby("_PARENT_BAND")[["_GLOBAL_PRO","_GLOBAL_S11"]].agg(["count","mean","std"])


Saved regression tables.
All done! Outputs saved in: /content/edu_analysis_outputs
